In [6]:
# Module 2 — RAG Search Engine
import json
from pathlib import Path

# Find the data file whether we're running from the project root OR the notebooks folder
data_path = Path.cwd() / "data" / "job_postings.json"
if not data_path.exists():
    data_path = Path.cwd().parent / "data" / "job_postings.json"

with open(data_path, "r", encoding="utf-8") as f:
    jobs = json.load(f)

print(f"Loaded {len(jobs)} job postings from:\n{data_path}\n")
print("Example posting:")
print(jobs[0])

Loaded 10 job postings from:
c:\Users\pallavi lakkoju\careerfit-ai\data\job_postings.json

Example posting:
{'title': 'Machine Learning Engineer', 'company': 'NovaAI', 'description': 'Build and deploy machine learning models in Python using PyTorch and scikit-learn. Work on NLP and recommendation systems.'}


In [7]:
import faiss
from sentence_transformers import SentenceTransformer

# Load the same embedding model as Module 1
model = SentenceTransformer("all-MiniLM-L6-v2")

# Combine each job's title + description into one piece of text
job_texts = [f"{job['title']}. {job['description']}" for job in jobs]

# Turn every job into a NORMALIZED vector (so inner product = cosine similarity)
job_vectors = model.encode(job_texts, normalize_embeddings=True)

# Build a FAISS index that searches by cosine similarity
dimension = job_vectors.shape[1]          # 384
index = faiss.IndexFlatIP(dimension)      # IP = Inner Product
index.add(job_vectors)

print(f"Indexed {index.ntotal} jobs, each as a {dimension}-dim vector.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 10 jobs, each as a 384-dim vector.


In [8]:
# A sample resume summary (later this will come from an uploaded PDF)
resume = ("Machine learning engineer experienced in Python, building and "
          "deploying predictive models with scikit-learn and PyTorch.")

# Embed the resume the SAME way (as a list, normalized)
resume_vector = model.encode([resume], normalize_embeddings=True)

# Ask FAISS for the top 3 most similar jobs
k = 3
scores, indices = index.search(resume_vector, k)

print("Top job matches for this resume:\n")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    job = jobs[idx]
    print(f"{rank}. {job['title']} — {job['company']}   (fit: {score:.2f})")
    print(f"   {job['description'][:80]}...\n")
    

Top job matches for this resume:

1. Machine Learning Engineer — NovaAI   (fit: 0.77)
   Build and deploy machine learning models in Python using PyTorch and scikit-lear...

2. Data Scientist — Insightly   (fit: 0.57)
   Analyze large datasets, build predictive models, and create dashboards. Strong P...

3. Deep Learning Researcher — DeepCore   (fit: 0.53)
   Research neural network architectures for computer vision. Experience with trans...

